In [249]:
import numpy as np
import pandas as pd
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Geometry import Point3D
import subprocess, re, shutil

In [250]:
SEED = 114514
HARTREE = 2625.5

In [251]:
def xyz_to_mol(smiles, path):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))      # 拓扑 + 氢（顺序：重原子，然后氢）
    lines = open(path).read().splitlines()            # 读整个文件
    n = int(lines[0])                                 # 第1行 = 原子数
    xyz = np.array([l.split()[1 : ] for l in lines[2 : 2 + n]], float)  # 取后3列 = 坐标
    conf = Chem.Conformer(n)                          # 建一个空构象
    for i, p in enumerate(xyz):
        conf.SetAtomPosition(i, Point3D(*p))          # 第 i 个原子 ← 第 i 行坐标
    mol.AddConformer(conf, assignId = True)           # 装进分子
    return mol

def frag_indices(mol, i, j):
    em = Chem.RWMol(mol)
    em.RemoveBond(i, j)
    return Chem.GetMolFrags(em, asMols = False)

def write_xyz(path, mol, idx):
    conf = mol.GetConformer()
    with open(path, "w") as f:
        f.write(f"{len(idx)}\n\n")
        for i in idx:
            a = mol.GetAtomWithIdx(i)
            p = conf.GetAtomPosition(i)
            f.write(f"{a.GetSymbol():2s}{p.x:16.8f}{p.y:16.8f}{p.z:16.8f}\n")

def run_xtb(folder, xyz, chrg, uhf, *flags):
    with open(folder / "xtb.out", "w") as f:
        p = subprocess.run([str(XTB), xyz, "--gfn", "2",
                            "--chrg", str(chrg), "--uhf", str(uhf),
                            *flags, "P", "4", "--json"],
                           cwd = folder, stdout = f, stderr = subprocess.STDOUT)
        return p.returncode

def read_energy(path):
    text = path.read_text(encoding = "utf-8")
    return float(re.search(r'"total energy"\s*:\s*(-?[\d.eE+-]+)', text).group(1))

JUNK = ("charges", "wbo", "xtbrestart", "xtbtopo.mol", "xtbout.json", "xtb.out", "xtbopt.xyz", "xtbopt.log", ".xtboptok")
def clean(folder):
    for f in folder.iterdir():
        if f.name in JUNK:
            f.unlink()

In [252]:
def bde_of_bond(mol, x, n, folder):
    frags = frag_indices(mol, x, n)
    if len(frags) != 2:
        return None
    f1, f2 = frags

    folder.mkdir(exist_ok = True)
    write_xyz(folder / "parent.xyz", mol, range(mol.GetNumAtoms()))
    write_xyz(folder / "fragA.xyz", mol, f1)
    write_xyz(folder / "fragB.xyz", mol, f2)

    if not (folder / "parent.json").exists():
        run_xtb(folder, "parent.xyz", 0, 0, "--scc")
        shutil.copy2(folder / "xtbout.json", folder / "parent.json")
        clean(folder)
    Ep = read_energy(folder / "parent.json")

    E_v = {}
    for name in ("fragA", "fragB"):                 # 垂直：片段保持父几何
        if not (folder / f"{name}_v.json").exists():
            run_xtb(folder, f"{name}.xyz", 0, 1, "--scc")
            shutil.copy2(folder / "xtbout.json", folder / f"{name}_v.json")
            clean(folder)
        E_v[name] = read_energy(folder / f"{name}_v.json")
    v = (E_v["fragA"] + E_v["fragB"] - Ep) * HARTREE
    return v

In [253]:
# 处理 .xlsx 数据，生成 mol 类
file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
df = df.sample(frac = 1, random_state = SEED, ignore_index = True)

STORE = Path(r"C:\Users\czywg\Desktop\Parents")
mols = []
for i in range(len(df)):
    mols.append(xyz_to_mol(df.loc[i, "SMILES"], STORE / f"parent_{i:04d}" / "conf.xyz"))
df["ROMol"] = mols
PandasTools.RenderImagesInAllDataFrames()

In [257]:
WORK = Path(r"C:\Users\czywg\Desktop\Programmer_czy\xtb_work\BDE")
XTB = Path(r"C:\Users\czywg\miniconda3\envs\xtb\Library\bin\xtb.exe")
FILE = Path(r"C:\Users\czywg\Desktop\Parents")

ALLPAT = [Chem.MolFromSmarts(p) for p in (
    "[#6]-[N+](=O)[O-]",          # C-NO2
    "[#7]-[N+](=O)[O-]",          # N-NO2
    "[#8]-[N+](=O)[O-]",          # O-NO2
)]

trigger_atoms = []

for cur, i in enumerate(sorted(FILE.glob("parent_*"))):
    mol = Chem.AddHs(df["ROMol"][cur])
    wbo = pd.read_csv(i / "wbo", sep = r"\s+", names=["a", "b", "wbo"])
    wbo_d = {(min(a, b) - 1, max(a, b) - 1) : w for a, b, w in wbo.values}

    for p in ALLPAT:
        minn = np.nan
        atoms = None
        for x, n, *_ in mol.GetSubstructMatches(p):
            w = wbo_d.get((min(x, n), max(x, n)))
            if w is not None and (minn != minn or w < minn):
                minn = w
                atoms = (x, n)
        trigger_atoms.append(atoms)

len(trigger_atoms)

4512

In [255]:
BDE_v = []

for i in df.index:
    sub = WORK / f"BDE_{i:04d}"
    sub.mkdir(exist_ok = True)

    vals = []
    for j, name in enumerate(["C-NO2", "N-NO2", "O-NO2"]):
        t_idx = i * 3 + j
        if trigger_atoms[t_idx] is None:
            continue
        v = bde_of_bond(df.loc[i, "ROMol"], trigger_atoms[t_idx][0], trigger_atoms[t_idx][1], sub / f"{name}")
        if v is not None:
            vals.append(v)
    
    BDE_v.append(min(vals) if vals else np.nan)
    pd.DataFrame({"BDE_v": BDE_v}).to_csv(WORK / "progress.csv", index = False)

print(len(BDE_v), pd.Series(BDE_v).isna().sum())

1504 202


In [256]:
df1 = pd.DataFrame({
    "Td": df["Td(°C)"],
    "BDE_v": BDE_v
})
print(df1.corr())

             Td     BDE_v
Td     1.000000  0.512312
BDE_v  0.512312  1.000000
